# Price a commodity storage contract

In [2]:
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d

In [8]:
df=pd.read_csv("Nat_Gas.csv")
df.head(10)

,Dates,Prices
0,10/31/20,10.10
1,11/30/20,10.30
2,12/31/20,11.00
3,1/31/21,10.90
4,2/28/21,10.90
5,3/31/21,10.90
6,4/30/21,10.40
7,5/31/21,9.84
8,6/30/21,10.00
9,7/31/21,10.10


In [9]:
df["Dates"] = pd.to_datetime(df["Dates"])
df = df.sort_values("Dates")
df.head()

C:\Users\sudee\AppData\Local\Temp\ipykernel_9296\2227231079.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Dates"] = pd.to_datetime(df["Dates"])


,Dates,Prices
0,2020-10-31,10.1
1,2020-11-30,10.3
2,2020-12-31,11.0
3,2021-01-31,10.9
4,2021-02-28,10.9


In [10]:
# Convert dates to numeric format for interpolation
date_numbers = df["Dates"].map(pd.Timestamp.toordinal)
prices = df["Prices"]

In [11]:
# Create interpolation function
price_model = interp1d(
    date_numbers,
    prices,
    kind="linear",
    fill_value="extrapolate"
)

In [12]:
# Function to estimate gas price on any date

In [13]:
def get_price(date):
    date = pd.to_datetime(date)
    return float(price_model(date.toordinal()))

In [16]:
# Storage Contract Pricing Function
def price_storage_contract(
    injection_dates,
    withdrawal_dates,
    volumes,
    max_storage,
    injection_rate,
    withdrawal_rate,
    storage_cost_per_month
):
    if not (
        len(injection_dates)
        == len(withdrawal_dates)
        == len(volumes)
    ):
        raise ValueError("Input lists must have the same length.")

    total_value = 0

    for inj_date, wd_date, volume in zip(
        injection_dates,
        withdrawal_dates,
        volumes
    ):

        if volume > max_storage:
            raise ValueError("Volume exceeds maximum storage capacity.")

        buy_price = get_price(inj_date)
        sell_price = get_price(wd_date)

        inj_date = pd.to_datetime(inj_date)
        wd_date = pd.to_datetime(wd_date)

        if wd_date <= inj_date:
            raise ValueError("Withdrawal date must be after injection date.")

        # Number of months in storage
        storage_months = (
            (wd_date.year - inj_date.year) * 12
            + (wd_date.month - inj_date.month)
        )

        # Trading profit
        revenue = sell_price * volume
        purchase_cost = buy_price * volume

        # Storage cost
        storage_cost = storage_months * storage_cost_per_month

        # Net profit
        contract_value = revenue - purchase_cost - storage_cost

        total_value += contract_value

    return total_value


# Example Test

In [19]:
injection_dates = [
    "2021-03-31",
    "2022-02-28"
]

withdrawal_dates = [
    "2021-09-30",
    "2022-10-31"
]

volumes = [
    1000,
    1500
]

max_storage = 5000
injection_rate = 500
withdrawal_rate = 500
storage_cost_per_month = 100

value = price_storage_contract(
    injection_dates,
    withdrawal_dates,
    volumes,
    max_storage,
    injection_rate,
    withdrawal_rate,
    storage_cost_per_month
)

print("Estimated Contract Value: $", round(value, 2))

Estimated Contract Value: $ -3300.0
